# ASTG-CodeBench: Quick Start & One-Click Reproducibility Demo

**A Standardized Spatial Topological Benchmark for Automated US Building Code Compliance Evaluation**

This interactive notebook demonstrates the deterministic compliance auditing of standard US multifamily apartment layouts against IBC 2024 (Egress), ADA 2010 (Accessibility), and LEED v4.1 (Daylight) criteria.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, Image, SVG

# Ensure source package is in path
import sys
src_path = str(Path("..").resolve() / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from astg_codebench.schemas.layout import LayoutTestCase
from astg_codebench.engine.oracle import audit
from astg_codebench.evaluate import load_cases, evaluate_oracle, print_report
from astg_codebench.render import render_case

print("ASTG-CodeBench modules imported successfully.")

## 1. Audit a Single Test Case (Non-Compliant Corridor: 43" vs 44" Required)

We load `CW-FAIL-01.json`, which represents a US 2BR apartment corridor where clear width is 43 inches (violating IBC 1020.2 for occupant load >= 50).

In [ ]:
case_path = Path("../data/v0.1/cases/CW-FAIL-01.json")
with open(case_path, "r", encoding="utf-8") as f:
    case = LayoutTestCase.model_validate(json.load(f))

result = audit(case)
print(f"Case ID: {result.case_id}")
print(f"Verdict: {result.label}")
print(f"Violations ({len(result.violations)}):")
for v in result.violations:
    print(f"  [{v.severity.value}] {v.rule_id} on {v.element_id}: {v.message}")

# Render visual diagram
fig = render_case(case, result)
plt.show()

## 2. Audit a Fully Compliant Apartment (Composite 1BR Suite)

Next, we audit `CMP-PASS-01.json`, an entire 1-bedroom apartment where all corridor widths, door maneuvering spaces, accessibility clearances, and window-to-floor ratios comply with all regulations.

In [ ]:
pass_case_path = Path("../data/v0.1/cases/CMP-PASS-01.json")
with open(pass_case_path, "r", encoding="utf-8") as f:
    pass_case = LayoutTestCase.model_validate(json.load(f))

pass_result = audit(pass_case)
print(f"Case ID: {pass_result.case_id}")
print(f"Verdict: {pass_result.label}")
print(f"Violations: {len(pass_result.violations)}")

fig = render_case(pass_case, pass_result)
plt.show()

## 3. Run the Full 36-Case Benchmark Evaluation

Here we execute the deterministic oracle across all 36 canonical test cases in the `v0.1` benchmark suite to compute the confusion matrix, Precision, Recall, and F1 Score.

In [ ]:
cases_dir = Path("../data/v0.1/cases")
cases = load_cases(cases_dir)
print(f"Loaded {len(cases)} test cases.")

metrics = evaluate_oracle(cases)
print_report(metrics)